In [3]:
!pip install -e .

Obtaining file:///Users/alemeneghini/Dropbox/stat/CPTM/260521_SCPTM/scptm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for scptm (pyproject.toml) ... done
  Created wheel for scptm: filename=scptm-0.2.0-0.editable-py3-none-any.whl size=7686 sha256=0255fa003c1c6eed7a9188b0893c3e3aa511ccb153f4a8de96876bdecaeaeac9
  Stored in directory: /private/var/folders/_8/fy4pl38d2tv85vx03yk7wlx00000gn/T/pip-ephem-wheel-cache-lr63n5a6/wheels/95/76/bb/5a7ea3a4a49f2cd771782040a829275073a503c1102f0d8fca
Successfully built scptm
  Attempting uninstall: scptm
    Found existing installation: scptm 0.2.0
    Uninstalling scptm-0.2.0:
      Successfully uninstalled scptm-0.2.0


In [ ]:
from scptm import SCPTM, SCPTMConfig
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from scptm.evaluation import compute_npmi_coherence, compute_topic_diversity
from scptm.graph import prepare_corpus
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

NOVELS_PATH = "/Users/alemeneghini/Dropbox/stat/CPTM/10_english_novels"   # ← aggiusta questo
CACHE       = "novels_cache.pkl"
K           = 20    # romanzi → più temi, prova 10-20

# ── Carica corpus una volta sola ─────────────────────────────────────────────
# apply_chunking=True: siccome lavoriamo con dati di romanzi sono lunghi, vengono spezzati in segmenti
# da ~800 caratteri (singoli paragrafi/scene) — unità più appropriate dei
# documenti interi per il topic modeling
docs = prepare_corpus(
    NOVELS_PATH,
    source_type   = "folder",
    apply_chunking = True,
    max_chunk_chars = 800,
)
print(f"Corpus: {len(docs)} segmenti da {len(set(docs))} file")

BASE = dict(
    num_topics      = K,
    lang            = "eng",
    epochs          = 50,
    apply_chunking  = False,   # già chunked sopra
    min_df          = 10,      # con ~3000+ segmenti, alza un po' il floor
    max_features    = 20_000,  # vocabolario letterario più ampio
    random_state    = 42,
)

# ── CTM baseline ─────────────────────────────────────────────────────────────
m_ctm = SCPTM(**BASE, graph_mode="none")
m_ctm.fit_transform(docs, edge_cache_path=CACHE)
r_ctm = m_ctm.evaluate()

# ── TriTopic-like ─────────────────────────────────────────────────────────────
m_tri = SCPTM(**BASE, graph_mode="none")
m_tri.fit(docs, edge_cache_path=CACHE,
          iterative_refinement=True, refinement_blend=0.2)
r_tri = m_tri.evaluate()

# ── SCPTM filtered ────────────────────────────────────────────────────────────
m_scptm = SCPTM(**BASE, graph_mode="filtered")
m_scptm.fit_transform(docs, edge_cache_path=CACHE)
r_scptm = m_scptm.evaluate()

# ── SCPTM + refine ────────────────────────────────────────────────────────────
m_best = SCPTM(**BASE, graph_mode="filtered")
m_best.fit(docs, edge_cache_path=CACHE,
           iterative_refinement=True, refinement_blend=0.2)
r_best = m_best.evaluate()

# ── BERTopic ──────────────────────────────────────────────────────────────────
sbert = SentenceTransformer("all-MiniLM-L6-v2")
bt = BERTopic(embedding_model=sbert, nr_topics=K,
              language="english", calculate_probabilities=True, verbose=True)
topics_bt, _ = bt.fit_transform(docs)

top_words_bt = [
    [w for w, _ in bt.get_topic(tid)[:10]]
    for tid in sorted(bt.get_topics()) if tid != -1
]
vec = CountVectorizer(min_df=10, max_features=20_000,
                      token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
bow_bt = vec.fit_transform(docs)
vocab_bt = vec.get_feature_names_out().tolist()
r_bt = {
    "npmi_coherence":  compute_npmi_coherence(top_words_bt, bow_bt, vocab_bt),
    "topic_diversity": compute_topic_diversity(top_words_bt),
}

# ── Tabella ───────────────────────────────────────────────────────────────────
rows = [
    ("CTM (no graph)",                r_ctm),
    ("TriTopic-like (no graph+refine)", r_tri),
    ("SCPTM (GNN filtered)",          r_scptm),
    ("SCPTM + refine",                r_best),
    ("BERTopic",                      r_bt),
]
df = pd.DataFrame([
    {"model":     name,
     "npmi":      round(r.get("npmi_coherence",  float("nan")), 3),
     "diversity": round(r.get("topic_diversity", float("nan")), 3)}
    for name, r in rows
])
print("\n", df.to_string(index=False))

Loaded 100 raw documents.
Final corpus: 109899 segments.
Corpus: 109899 segmenti da 109899 file


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7799.48it/s]


NLP pipeline: SBERT='all-MiniLM-L6-v2' on cpu, spaCy='en_core_web_sm' on CPU
Loaded 109899 raw documents.

[Graph] Mode: 'none' — No syntactic graph — equivalent to CTM + KL annealing
1/5  Building lemma vocabulary...


Lemmatisation: 100%|██████████| 109898/109898 [24:49<00:00, 73.76it/s]


  Vocabulary: 20000 unique lemmas.
2/5  Encoding documents (SBERT)...


...

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8042.54it/s]


NLP pipeline: SBERT='all-MiniLM-L6-v2' on cpu, spaCy='en_core_web_sm' on CPU
Loaded 109899 raw documents.

[Graph] Mode: 'none' — No syntactic graph — equivalent to CTM + KL annealing
  [EdgeCache] Loaded from 'novels_cache.pkl' — skipping spaCy parsing.
  Vocabulary: 20000 unique lemmas (from cache).
2/5  Document embeddings loaded from cache.
  MODE 'none': no edges generated (CTM-like baseline).
  Word embeddings loaded from cache.

[Memory estimate]
  node_features_MB         : 190.3
  edge_index_MB            : 0.0
  activations_MB           : 570.8
  gradients_MB             : 380.6
  total_estimated_GB       : 1.11
  [CtxCache] Loaded from 'novels_cache.pkl' — skipping contextual embedding pass.
  Topic embeddings initialised from word-embedding k-means (mean top-sim per word: 0.533)

[Iterative Refinement] Step 1/2

Training — mode=none, device=cpu
  Mixed precision: True | Neighbor sampling: False
Epoch 010/50  Loss=9.897  Recon=8.859  KL=2.002  KL-w=0.500  NPMI=-0.184  Div=0.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8205.07it/s]


NLP pipeline: SBERT='all-MiniLM-L6-v2' on cpu, spaCy='en_core_web_sm' on CPU
Loaded 109899 raw documents.

[Graph] Mode: 'filtered' — Informative dependency types only (default)


: 

In [ ]:
from tritopic import TriTopic
from scptm.evaluation import compute_npmi_coherence, compute_topic_diversity
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd, numpy as np

K = 20   # stesso numero di topic degli altri modelli

# ── TriTopic ──────────────────────────────────────────────────────────────────
tri = TriTopic(
    n_topics          = K,
    embedding_model   = "all-MiniLM-L6-v2",   # stesso SBERT di SCPTM/BERTopic
    random_state      = 42,
    verbose           = True,
)
tri.fit_transform(docs)

# Metriche native (coherence calcolata sui soli doc del topic → punteggio più alto)
r_tri_native = tri.evaluate()

# Metriche ricalcolate con la stessa funzione usata da SCPTM (full-corpus BoW)
# → confronto fair con tutti gli altri modelli
top_words_tri = [
    t.keywords[:10]
    for t in tri.topics_
    if t.topic_id != -1
]
vec = CountVectorizer(
    min_df=10, max_features=20_000,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)
bow_common = vec.fit_transform(docs)
vocab_common = vec.get_feature_names_out().tolist()

npmi_tri_fair = compute_npmi_coherence(top_words_tri, bow_common, vocab_common)
div_tri_fair  = compute_topic_diversity(top_words_tri)


[TriTopic] Fitting model on 11244 documents
   Config: hybrid graph, iterative mode
   > Generating embeddings (all-MiniLM-L6-v2)...


Batches: 100%|██████████| 352/352 [01:20<00:00,  4.38it/s]


   > Reducing dimensions to 10d (umap)...
   > Building lexical similarity matrix...
   > Starting iterative refinement (max 5 iterations)...
      Iteration 1...
      Iteration 2...
         ARI vs previous: 0.9504
      Converged at iteration 2
   > Extracting keywords and representative documents...

   > Auto-tuning resolution for 15 topics (currently 33)...
      Found resolution=0.313
Reducing from 16 to 15 topics...
   Now have 15 topics.
      Final topic count: 15

[OK] Fitting complete!
   Found 15 topics
   0 outlier documents (0.0%)

[Metrics] Evaluation:
   Coherence (mean): 0.0415
   Diversity: 0.8800
   Stability: 0.9211
   Outlier ratio: 0.00%
